# Interview simulator

### ASR test in 4 wariants:

1. raw Whisper (baseline)
2. Whisper with input conifguration (`hotwords` and `initial_prompt`)
3. Whisper asr with dictionary based correction
4. Whisper asr with dictionary and SLM based corrections


### Configuration:

Core project conifguration can be changed in `src/constatns.py`


In [4]:
import sys
from pathlib import Path

SRC_DIR = Path.cwd().parent / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

In [5]:
from constants import RUNS_DIR
from log_runs import AudioSplit, RunConfig, WhisperConfig, FileRunRepository
from project_data import ProjectRepository

whisper_config = WhisperConfig(
    model="large-v3-turbo",
    device="cpu",
    compute_type="int8",
    language="pl",
    beam_size=5,
    temperature=0.0,
    vad_filter=True,
)

config = RunConfig(
    variant=1,
    whisper=whisper_config,
)

run_repo = FileRunRepository(RUNS_DIR)
project_repo = ProjectRepository()
split: AudioSplit = "dev"

### First wariant:


In [ ]:
from run_variant import run_variant_first

run_id = run_variant_first(
    config=config,
    split=split,
    run_repo=run_repo,
    project_repo=project_repo,
)

### Variant 2: hotwords token check

In [3]:
import pandas as pd
from faster_whisper import WhisperModel
from faster_whisper.tokenizer import Tokenizer
from process import build_hotwords, count_hotword_tokens, get_dropped_hotwords
from constants import DICTIONARY_PATH

# TODO (#12): Replace this block with project.get_dictionary()
dictionary = pd.read_csv(DICTIONARY_PATH, keep_default_na=False)

hidden_terms = [
    "Jenkins", "Microsoft Teams",
    "Redis", "Azure",
    "Scrum", "sprint",
    "deploy", "merge",
    "POC", "API",
]

dictionary = dictionary[
    ~dictionary["term"].str.lower().isin(
        [term.lower() for term in hidden_terms]
    )
]

hotwords = build_hotwords(dictionary)

model = WhisperModel(
    whisper_config.model,
    device=whisper_config.device,
    compute_type=whisper_config.compute_type,
)
tokenizer = Tokenizer(
    model.hf_tokenizer,
    model.model.is_multilingual,
    task="transcribe",
    language=whisper_config.language,
)

n_tokens = count_hotword_tokens(hotwords, tokenizer)
limit = model.max_length // 2 - 1
dropped = get_dropped_hotwords(hotwords, tokenizer, model.max_length)

prompt = model.get_prompt(tokenizer, [], hotwords=" ".join(hotwords))
in_prompt = len(prompt) - 1 - len(tokenizer.sot_sequence)

print(f"terms:      {len(hotwords)}")
print(f"tokens:     {n_tokens}")
print(f"limit:      {limit}")
print(f"truncated:  {n_tokens > limit}")
print(f"dropped:    {dropped}")
print(f"cross-check (library keeps {in_prompt} tokens): {in_prompt == min(n_tokens, limit)}")

terms:      84
tokens:     155
limit:      223
truncated:  False
dropped:    []
cross-check (library keeps 155 tokens): True
